# Tree Crown Segmentation with PyCrown

Delineates individual tree crowns from AHN4 LiDAR-derived rasters using the PyCrown
library. The resulting crown polygons are exported as a shapefile.

**Import libraries**

In [ ]:
import os
import sys
from pathlib import Path
import rasterio
from pycrown import PyCrown

# Locate project root
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Enable imports from src
sys.path.append(str(PROJECT_ROOT))

from src.paths import RAW_DIR, EXTERNAL_DIR, INTERIM_DIR, PROCESSED_DIR

**Define paths**

In [ ]:
# LiDAR-derived rasters produced by the CHM creation workflow.
# All three files must share the same extent, resolution, and CRS (EPSG:28992).
# CHM and DSM are at 0.25 m resolution (resampled to match RGB imagery);
# DTM is at 0.5 m (IDW-interpolated, merged, and aligned to the DSM extent).
F_CHM = PROCESSED_DIR / "lidar" / "chm_0p25.tif"
F_DTM = PROCESSED_DIR / "lidar" / "dtm_interp_match_dsm.tif"
F_DSM = PROCESSED_DIR / "lidar" / "dsm_resol.tif"

# Output directory where PyCrown writes the CHM, treetop, and crown shapefiles
OUTPATH = INTERIM_DIR / "pycrown_segmentation" 

**Initialise PyCrown**

In [ ]:
# PyCrown loads all three rasters and prepares the processing environment.
# The outpath directory is created automatically if it does not exist.
PC = PyCrown(F_CHM, F_DTM, F_DSM, outpath=OUTPATH)

**Clip all input data to the raster bounding box**

In [ ]:
# Restrict processing to the shared extent of the tile (34FN2).
# Coordinates are in RD New (EPSG:28992): (xmin, xmax, ymin, ymax).
PC.clip_trees_to_bbox(
    bbox=(254999.504, 260000.004, 468749.707, 475000.207)
)

**Smooth the CHM**

In [ ]:
# Apply a 5-pixel Gaussian smoothing filter to reduce point-cloud noise before
# treetop detection. Smoothing prevents spurious local maxima inside crown
# interiors from being detected as separate trees.
PC.filter_chm(5, ws_in_pixels=True, circular=False)

**Detect treetops**

In [ ]:
# Detect treetops using a local maximum filter with a 5-pixel search window.
# hmin=7 m discards low vegetation (shrubs, hedges) that is not of interest.
PC.tree_detection(PC.chm, ws=5, hmin=7)
print("Detected treetops:", len(PC.trees))

**Clip detected trees to the inset bounding box**

In [ ]:
# Remove treetops within 5 m of the tile edge. Edge trees are often truncated
# by the tile boundary, producing incomplete and unreliable crown polygons.
margin = 5.0
PC.clip_trees_to_bbox(
    bbox=(
        254999.504 + margin,   # xmin
        260000.004 - margin,   # xmax
        468749.707 + margin,   # ymin
        475000.207 - margin    # ymax
    )
)
print("Treetops after edge clip:", len(PC.trees))

**Delineate tree crowns**

In [ ]:
# Crown delineation uses the Dalponte region-growing algorithm (numba-accelerated).
#
# th_tree  = 3.0 m  — minimum CHM height for crown growth; pixels below this
#                      threshold cannot be assigned to any crown.
# th_seed  = 0.6    — a pixel is accepted as part of a crown only if its CHM
#                      value is at least 60 % of the treetop height.
# th_crown = 0.40   — controls lateral crown expansion; pixels up to 40 % below
#                      the treetop height may still be included in the crown.
# max_crown= 10.0 m — caps the maximum crown radius to prevent unrealistically
#                      large delineations.
PC.crown_delineation(
    algorithm='dalponteCIRC_numba',
    th_tree=3.0,
    th_seed=0.6,
    th_crown=0.40,
    max_crown=10.0,
)
print("Delineated crowns:", len(PC.trees))

**Calculate tree height and elevation**

In [ ]:
# Record the CHM height and DTM elevation at each treetop position.
# 'top'     — raw treetop pixel position.
# 'top_cor' — corrected treetop position (used when screen_small_trees is applied;
#             set to 0.0 below because that step is skipped here).
PC.get_tree_height_elevation(loc='top')
PC.get_tree_height_elevation(loc='top_cor')

**Convert raster crowns to polygons**

In [ ]:
# tt_corrected is required by crowns_to_polys_raster() but is only populated
# by screen_small_trees(), which is not used in this workflow. Setting it to
# 0.0 satisfies the API without altering the delineation results.
PC.trees["tt_corrected"] = 0.0

# Vectorise the raster crown labels to polygon geometries.
PC.crowns_to_polys_raster()

**Quality control**

In [ ]:
# Inspect the first rows of the tree attribute table.
PC.trees.head()

In [ ]:
# Check that all crown polygons have valid geometries (no self-intersections,
# empty rings, etc.). PyCrown prints a summary of any issues found.
PC.quality_control()

**Export results**

In [ ]:
print(f"Total delineated tree crowns: {len(PC.trees)}")

# Write the crown polygon shapefile to OUTPATH. 
PC.export_tree_crowns(crowntype='crown_poly_raster')

**Post-processing note**

After exporting, the crown polygon shapefile was opened in QGIS and over-segmented
polygons were removed. The filtered shapefile was saved as
`enschede_pycrown_segmentation_tile34fn2_v01.shp` and is used as input for the ground-truth preprocessing
notebook ("crown_species_labelling.ipynb`).